# Static Runtime Context

**Static runtime context** is immutable data passed to your agent at the start of a run. Think of it as **dependency injection** for your agent.

Typical examples:
- User metadata (`user_id`, `user_name`, `email`)
- Database connections / API clients
- Tools that depend on the current user or session

It's passed via the `context` argument to `invoke()` / `stream()` and **does not change** during execution.

> Source: [LangChain Context docs](https://docs.langchain.com/oss/python/concepts/context#static-runtime-context)

## Why not just hardcode it?

Without static context, you'd have to bake values into your tools or prompts:

```python
@tool
def get_user_email() -> str:
    return get_user_email_from_db("john_smith")  # hardcoded 🙁
```

With static context, the same agent works for any user — the value is passed at runtime:

```python
agent.invoke({...}, context=ContextSchema(user_name="john_smith"))
agent.invoke({...}, context=ContextSchema(user_name="jane_doe"))
```

## Setup

Make sure you have a `.env` file in your project directory with:

```
OPENAI_API_KEY=your-api-key-here
OPENAI_ENDPOINT=your-endpoint-here
```

We load it, verify the env vars, and instantiate the chat model — then pass the `llm` object to `create_agent`.

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

assert os.environ.get("OPENAI_API_KEY"), "OPENAI_API_KEY is not set! Check your .env file."
assert os.environ.get("OPENAI_ENDPOINT"), "OPENAI_ENDPOINT is not set! Check your .env file."

from dataclasses import dataclass
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from langchain.tools import tool, ToolRuntime

llm = ChatOpenAI(base_url=os.environ["OPENAI_ENDPOINT"], model="gpt-5.4-mini")

print("Environment variables loaded successfully!")

## Step 1: Define a context schema

The schema is just a `dataclass` describing what static data the agent expects.

In [ ]:
@dataclass
class ContextSchema:
    user_name: str

## Step 2: Read the context inside a tool

A tool receives a `ToolRuntime[ContextSchema]` parameter. Access static data via `runtime.context`.

In [ ]:
def get_user_email_from_db(user_name: str) -> str:
    # Pretend this hits a real database.
    return f"{user_name}@example.com"


@tool
def get_user_email(runtime: ToolRuntime[ContextSchema]) -> str:
    """Retrieve the current user's email."""
    return get_user_email_from_db(runtime.context.user_name)

## Step 3: Register the schema with the agent

Pass `context_schema=ContextSchema` when creating the agent so it knows what shape to expect.

In [ ]:
agent = create_agent(
    model=llm,
    tools=[get_user_email],
    context_schema=ContextSchema,
)

## Step 4: Pass context at invoke time

Notice the same agent can answer for *any* user — just change the `context`.

In [ ]:
response = agent.invoke(
    {"messages": [{"role": "user", "content": "What is my email address?"}]},
    context=ContextSchema(user_name="john_smith"),
)

for message in response["messages"]:
    message.pretty_print()

In [ ]:
# Same agent, different user — no code changes needed.
response = agent.invoke(
    {"messages": [{"role": "user", "content": "What is my email address?"}]},
    context=ContextSchema(user_name="jane_doe"),
)

for message in response["messages"]:
    message.pretty_print()

## Bonus: use static context in the system prompt

With a `@dynamic_prompt` middleware, the agent can personalize its system prompt based on context.

In [ ]:
from langchain.agents.middleware import dynamic_prompt, ModelRequest


@dynamic_prompt
def personalized_prompt(request: ModelRequest) -> str:
    user_name = request.runtime.context.user_name
    return f"You are a helpful assistant. Always address the user as {user_name}."


agent = create_agent(
    model=llm,
    tools=[get_user_email],
    middleware=[personalized_prompt],
    context_schema=ContextSchema,
)

response = agent.invoke(
    {"messages": [{"role": "user", "content": "Hi! Say hello."}]},
    context=ContextSchema(user_name="John Smith"),
)

for message in response["messages"]:
    message.pretty_print()

## Static context in LangGraph

So far we've used `create_agent` (LangChain). The same idea works in a raw **LangGraph** workflow — you build a `StateGraph` and any node can receive a `Runtime[ContextSchema]` parameter to read the static context.

The pattern:

1. Define a state schema (the *mutable* data flowing through nodes).
2. Define a context schema (the *immutable* data for this run).
3. Each node has the signature `def node(state, runtime: Runtime[ContextSchema])`.
4. Pass `context=...` to `graph.invoke()`, just like with the agent.

In [ ]:
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.runtime import Runtime


class State(TypedDict):
    greeting: str


def greet_node(state: State, runtime: Runtime[ContextSchema]) -> State:
    user_name = runtime.context.user_name  # <-- static context
    return {"greeting": f"Hello, {user_name}! Welcome back."}


builder = StateGraph(State, context_schema=ContextSchema)
builder.add_node("greet", greet_node)
builder.add_edge(START, "greet")
builder.add_edge("greet", END)
graph = builder.compile()

result = graph.invoke(
    {"greeting": ""},
    context=ContextSchema(user_name="john_smith"),
)

print(result["greeting"])

Same `ContextSchema`, same `context=...` argument — just a different runtime object (`Runtime` for nodes vs `ToolRuntime` for tools).

## Try it yourself 🛠️

Extend the `ContextSchema` to also include a `language` field (e.g. `"en"`, `"de"`, `"es"`).

Then:

1. Update the `dynamic_prompt` so the assistant replies in the user's preferred language.
2. Invoke the agent with a context like `ContextSchema(user_name="Maria", language="es")` and ask `"Tell me a fun fact about cats."`

**Hint:** you only need to change the dataclass and the prompt string.

In [ ]:
# Your solution here

@dataclass
class ContextSchema:
    user_name: str
    # TODO: add a `language` field


@dynamic_prompt
def personalized_prompt(request: ModelRequest) -> str:
    # TODO: read user_name AND language from request.runtime.context
    # TODO: return a system prompt that tells the model to address the user
    #       by name AND reply in their preferred language.
    ...


agent = create_agent(
    model=llm,
    tools=[get_user_email],
    middleware=[personalized_prompt],
    context_schema=ContextSchema,
)

response = agent.invoke(
    {"messages": [{"role": "user", "content": "Tell me a fun fact about cats."}]},
    # TODO: pass a context with user_name="Maria" and language="es"
)

for message in response["messages"]:
    message.pretty_print()